# CarveFormer — FFT-75 Benchmark (Kaggle)

Trains and evaluates **CarveFormer** (Swin Transformer V2 Tiny + 96-d byte
embedding) on FFT-75, using the shared DeepCarv framework.

**Data comes from attached Kaggle datasets — no `gdown`.**
Attach both:
* `thegifman/fft-75-512-1`
* `thegifman/fft-75-4096-1`

Why not gdown: downloading a zip and extracting it keeps *both* copies on disk
and pulls the arrays through RAM, which overflows the Kaggle instance. Kaggle
inputs are already extracted, and the loader **memory-maps** them, so a 25 GB
training split costs ~0 RAM.

Switch benchmarks by editing `FRAGMENT_SIZE` in §0 only.

## §0 — Configuration (edit here only)

In [ ]:
# The one switch.
FRAGMENT_SIZE = 512      # 512 or 4096
NUM_CLASSES   = 75       # FFT-75 Scenario #1
PRETRAINED    = True     # ImageNet1k init (paper default)

# --- Throughput ---------------------------------------------------------------
BATCH_SIZE    = 64       # CarveFormer is ~28M params; 64 fits a T4
EPOCHS        = 50       # paper
NUM_WORKERS   = 4        # data is mmapped off disk now -> parallel workers matter
AMP           = True     # mixed precision: ~1.5-2x on a T4, big win

# --- Session timeout ----------------------------------------------------------
# Kaggle kills the session at 12h. Rather than being killed MID-EPOCH (losing
# that epoch's work), we stop cleanly at an epoch boundary before the limit.
# The run checkpoints every epoch, so you just RE-RUN THE TRAINING CELL in a
# fresh session and it resumes exactly where it stopped.
MAX_HOURS     = 11.0     # leave ~1h for evaluation + artifact upload

print(f'fragment={FRAGMENT_SIZE} batch={BATCH_SIZE} epochs={EPOCHS} '
      f'workers={NUM_WORKERS} amp={AMP} budget={MAX_HOURS}h')

## §1 — Environment + repo

In [ ]:
import os, sys, gc, shutil
from pathlib import Path

!pip install -q "timm>=1.0.0" pyyaml >/dev/null 2>&1

WORKING  = Path('/kaggle/working')
REPO_DIR = WORKING / 'deepcarv'
BRANCH   = 'memory-fix-and-benchmarks'   # branch containing the mmap dataset fix

if not (REPO_DIR / 'src').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    os.system(f'git clone --depth 1 --branch {BRANCH} https://github.com/yuvnahr/deepcarv.git {REPO_DIR}')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## §2 — Data (Kaggle inputs, memory-mapped, no gdown)

In [ ]:
# =============================================================================
# DATA — Kaggle inputs (NO gdown)
# =============================================================================
# The FFT-75 splits are attached as Kaggle Datasets:
#     thegifman/fft-75-512-1   -> /kaggle/input/fft-75-512-1
#     thegifman/fft-75-4096-1  -> /kaggle/input/fft-75-4096-1
#
# We deliberately do NOT use gdown. Downloading a zip and unzipping it keeps
# BOTH copies on disk and pulls the arrays through RAM, which is what was
# blowing up the Kaggle instance. Kaggle inputs are already extracted and are
# read straight off disk.
#
# We also do NOT copy/re-save the npz files into /kaggle/working. The dataset
# loader memory-maps them in place (src/data/npz_mmap.py), so a 25 GB training
# split costs ~0 RAM — only the batches actually read are paged in.
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')

# Map fragment size -> mounted Kaggle dataset directory.
DATASET_DIRS = {
    512:  KAGGLE_INPUT / 'fft-75-512-1',
    4096: KAGGLE_INPUT / 'fft-75-4096-1',
}


def resolve_split_dir(fragment_size: int) -> Path:
    """Find the directory holding train/val/test.npz for a fragment size.

    Kaggle datasets sometimes mount with an extra nesting level, so we search
    rather than assume a fixed depth.
    """
    root = DATASET_DIRS[fragment_size]
    if not root.exists():
        raise FileNotFoundError(
            f'Kaggle input not attached: {root}\n'
            f'Add the dataset to this notebook (Add Data -> search "fft-75").'
        )
    candidates = [root, *[p for p in root.rglob('*') if p.is_dir()]]
    for cand in candidates:
        if (cand / 'train.npz').exists():
            return cand
    raise FileNotFoundError(f'No train.npz found anywhere under {root}')


def build_data_root(fragment_size: int) -> Path:
    """Return a root dir laid out as {root}/{fragment_size}/{split}.npz.

    The framework expects that layout. Rather than COPYING the npz files (which
    would duplicate tens of GB), we symlink them — zero extra bytes, and mmap
    still works through the link.
    """
    split_dir = resolve_split_dir(fragment_size)
    data_root = Path('/kaggle/working/data/FFT-75')
    target = data_root / str(fragment_size)
    target.mkdir(parents=True, exist_ok=True)

    for split in ('train', 'val', 'test'):
        src = split_dir / f'{split}.npz'
        if not src.exists():
            raise FileNotFoundError(f'Missing split: {src}')
        dst = target / f'{split}.npz'
        if dst.is_symlink() or dst.exists():
            dst.unlink()
        dst.symlink_to(src)          # link, never copy
        size_gb = src.stat().st_size / 1e9
        print(f'  {fragment_size}/{split}.npz -> {src.name} ({size_gb:.2f} GB, symlinked)')

    return data_root


DATA_ROOT = build_data_root(FRAGMENT_SIZE)
print(f'\nDATA_ROOT = {DATA_ROOT}  (fragment_size={FRAGMENT_SIZE})')

# Confirm the loader will memory-map these (i.e. RAM stays flat).
from src.data.npz_mmap import is_mmappable

for split in ('train', 'val', 'test'):
    p = DATA_ROOT / str(FRAGMENT_SIZE) / f'{split}.npz'
    print(f'  {split}: mmappable={is_mmappable(p)}')
print(
    '\nNOTE: mmappable=False means that file was written with savez_compressed.\n'
    '      It will still load, but via RAM. Re-save it uncompressed with\n'
    '      src.data.npz_mmap.rewrite_uncompressed() to get the memory benefit.'
)


## §3 — Sanity check

A couple of hundred steps to prove the wiring works before spending GPU hours.

In [ ]:
import yaml, gc
from benchmarks.CarveFormer.scripts.train import run_training

CONFIG_PATH = REPO_DIR / 'benchmarks/CarveFormer/configs/benchmark.yaml'
config = yaml.safe_load(open(CONFIG_PATH))

config['dataset']['root_dir']      = str(DATA_ROOT)
config['dataset']['fragment_size'] = FRAGMENT_SIZE
config['dataset']['num_classes']   = NUM_CLASSES
config['dataset']['mmap']          = True       # keep RAM flat
config['model']['pretrained']      = PRETRAINED
config['training']['batch_size']   = BATCH_SIZE
config['training']['epochs']       = EPOCHS
config['training']['num_workers']  = NUM_WORKERS
config['training']['amp']          = AMP
config['training']['max_hours']    = MAX_HOURS  # stop cleanly before the 12h kill
config['paths']['run_outputs']     = str(WORKING / f'carveformer_{FRAGMENT_SIZE}')

RUN_DIR = Path(config['paths']['run_outputs'])
print('run dir:', RUN_DIR)

## §4 — Full training

Paper hyperparameters (AdamW, lr 3.75e-4, wd 0.05, 50 epochs) come from the
committed config. Note the batch-size deviation below.

In [ ]:
# ---------------------------------------------------------------------------
# RESUMABLE TRAINING
# ---------------------------------------------------------------------------
# Safe to re-run. run_training() -> Trainer.fit_or_resume(), which:
#   * picks up checkpoint_last.pt automatically if it exists,
#   * restores the LR schedule, best-score tracking, early-stopping counters,
#     and the full history (not just the weights),
#   * stops cleanly at MAX_HOURS instead of being killed mid-epoch,
#   * no-ops if training already reached EPOCHS.
#
# If the session times out: start a new session, run the cells above, then
# re-run THIS cell. It continues from the last completed epoch.
ckpt = RUN_DIR / 'checkpoint_last.pt'
if ckpt.exists():
    import torch
    done = torch.load(ckpt, map_location='cpu', weights_only=False).get('epoch', 0)
    print(f'Found checkpoint at epoch {done}/{EPOCHS} -> resuming.\n')
else:
    print('No checkpoint found -> starting from scratch.\n')

run_dir = run_training(config)
gc.collect()
print('\nTraining call finished ->', run_dir)
print('If the log says the time budget was reached, re-run this cell in a '
      'fresh session to continue.')

## §5 — Results

In [ ]:
import json
run_dir = Path(config['paths']['run_outputs'])

summary = json.load(open(run_dir / 'summary.json'))
print(json.dumps(summary, indent=2))

# Compare against the paper.
PAPER = {512: 0.7210, 4096: 0.8299}   # CarveFormer, FFT-75 Scenario #1
acc = summary.get('accuracy')
target = PAPER[FRAGMENT_SIZE]
print(f"\nCarveFormer @ {FRAGMENT_SIZE}B")
print(f"  ours : {acc:.4f}")
print(f"  paper: {target:.4f}")
print(f"  gap  : {acc - target:+.4f}")

## §6 — Save outputs

Archive the standardized output set (metrics.json, summary.json,
predictions.csv, confusion_matrix.csv, per_class_metrics.csv,
classification_report.txt) for the results write-up.

In [ ]:
archive = shutil.make_archive(
    str(WORKING / f'carveformer_fft75_{FRAGMENT_SIZE}'), 'zip', run_dir
)
print('Archive:', archive)
for p in sorted(run_dir.glob('*')):
    print(' ', p.name)